# ClaimGuard — session 5b: latency of the LLM judge (about 15 min)

Session 5 timed every system except the Qwen-7B judge (bitsandbytes was missing on the Kaggle image).
This notebook installs it and times the judge and, in the same session, the main system again.
Same inputs as session 5. GPU T4 x2, Internet on. *Save Version → Save & Run All*.

## 0. Setup

In [ ]:
import glob, json, os, shutil, subprocess, time, urllib.request
from pathlib import Path

# several versions of the code can be mounted: take the one whose session5.py has --only
candidates = [Path(p).parent for p in glob.glob("/kaggle/input/**/scripts/session5.py", recursive=True)
              if "--only" in Path(p).read_text()]
candidates = [p.parent for p in candidates]
assert candidates, "claimguard_v5b.zip not found: add the new version of claimguard-code as input"
SRC = candidates[0]
WORK = Path("/kaggle/working/claimguard")
if not WORK.exists():
    shutil.copytree(SRC, WORK)
os.chdir(WORK)
assert (WORK / "scripts/session5.py").exists(), "old claimguard.zip: upload the new version"
print("project copied from", SRC)

In [ ]:
!pip install -q -r requirements.txt && pip install -q -e .
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU: set the accelerator to GPU T4 x2")

In [ ]:
# run a shell command, log time and success, never stop the notebook
STATUS_FILE = WORK / "results" / "run_status_multihop.json"
STATUS_FILE.parent.mkdir(exist_ok=True)
status = []

def run(name, cmd):
    print(f"\n===== {name} =====\n$ {cmd}", flush=True)
    start = time.time()
    proc = subprocess.run(cmd, shell=True, cwd=WORK, text=True, capture_output=True)
    minutes = round((time.time() - start) / 60, 1)
    print(proc.stdout[-4000:])
    if proc.returncode != 0:
        print("STDERR:", proc.stderr[-4000:])
    status.append({"step": name, "ok": proc.returncode == 0, "minutes": minutes,
                   "error": None if proc.returncode == 0 else proc.stderr[-1500:]})
    STATUS_FILE.write_text(json.dumps(status, indent=2))
    print(f"-> {'OK' if proc.returncode == 0 else 'FAILED'} in {minutes} min", flush=True)
    return proc.returncode == 0

In [ ]:
# my SciFact verifier (session 2)
import zipfile
zips = glob.glob("/kaggle/input/**/verifier.zip", recursive=True)
dirs = [Path(p).parent for p in glob.glob("/kaggle/input/**/verifier/config.json", recursive=True)]
if zips:
    zipfile.ZipFile(zips[0]).extractall(WORK / "models")
elif dirs:
    shutil.copytree(dirs[0], WORK / "models" / "verifier", dirs_exist_ok=True)
assert (WORK / "models/verifier/config.json").exists(), "verifier not found: add the session-2 output or verifier.zip"
print("verifier OK")

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
    print("Hugging Face login OK")
except Exception as err:
    print("no HF_TOKEN secret (only needed if the Qwen download is gated):", type(err).__name__)
STATUS_FILE = WORK / "results" / "run_status_session5.json"
run("data", "make data")

In [ ]:
!pip install -q -U "bitsandbytes>=0.46.1" accelerate

In [ ]:
run("latency GPU: main + LLM judge", "python scripts/session5.py latency --config configs/default.yaml --n 20 --only main llm_judge")
print((WORK / "results/latency_cuda.json").read_text())
shutil.copy(WORK / "results/latency_cuda.json", "/kaggle/working/latency_llm_judge.json")